In [19]:
import random
from collections import deque

GRID_SIZE = 10
NUM_DOGS = 3
NUM_FOOD = 8
MAX_STEPS = 100
RANDOM_SEED = None


class Environment:
    def __init__(self, size, num_food):
        self.size = size
        self.food = set()
        while len(self.food) < num_food:
            pos = (random.randint(0, size - 1), random.randint(0, size - 1))
            self.food.add(pos)

    def is_food(self, pos):
        return pos in self.food

    def remove_food(self, pos):
        self.food.discard(pos)


class Dog:
    def __init__(self, name, pos):
        self.name = name
        self.pos = pos
        self.score = 0
        self.path = []

    def bfs_nearest_food(self, env):
        if not env.food:
            return None

        start = self.pos
        visited = {start}
        queue = deque([(start, [])])

        while queue:
            current, path = queue.popleft()
            if current in env.food:
                return path
            x, y = current
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                npos = (x + dx, y + dy)
                if (
                    0 <= npos[0] < env.size
                    and 0 <= npos[1] < env.size
                    and npos not in visited
                ):
                    visited.add(npos)
                    queue.append((npos, path + [npos]))
        return None

    def step(self, env):
        if self.path and self.path[-1] not in env.food:
            self.path = []

        if not self.path:
            self.path = self.bfs_nearest_food(env) or []

        if self.path:
            self.pos = self.path.pop(0)
            if env.is_food(self.pos):
                env.remove_food(self.pos)
                self.score += 1
                self.path = []


def print_grid(env, dogs):
    grid = [["." for _ in range(env.size)] for _ in range(env.size)]

    for fx, fy in env.food:
        grid[fx][fy] = "F"

    for dog in dogs:
        x, y = dog.pos
        grid[x][y] = dog.name[-1]

    for row in grid:
        print(" ".join(row))
    print()


def main():
    if RANDOM_SEED is not None:
        random.seed(RANDOM_SEED)

    env = Environment(GRID_SIZE, NUM_FOOD)
    dogs = [
        Dog(f"Dog{i + 1}", (random.randint(0, GRID_SIZE - 1), random.randint(0, GRID_SIZE - 1)))
        for i in range(NUM_DOGS)
    ]

    print("Start")
    print(f"Grid: {GRID_SIZE}x{GRID_SIZE} | สุนัข: {NUM_DOGS} ตัว | อาหาร: {NUM_FOOD} ชิ้น\n")
    print_grid(env, dogs)

    step_count = 0
    while env.food and step_count < MAX_STEPS:
        step_count += 1
        for dog in dogs:
            dog.step(env)

        print(f"--- Step {step_count} ---")
        print_grid(env, dogs)

    print("End")
    print(f"ใช้เวลาทั้งหมด {step_count} steps\n")

    for dog in sorted(dogs, key=lambda d: -d.score):
        print(f"{dog.name}: เก็บอาหารได้ {dog.score} ชิ้น")

    winner = max(dogs, key=lambda d: d.score)
    print(f"\n Winner: {winner.name} ({winner.score} ชิ้น)")


if __name__ == "__main__":
    main()

Start
Grid: 10x10 | สุนัข: 3 ตัว | อาหาร: 8 ชิ้น

. . . F . . . . . .
. . . . 1 . . . . .
. . . . . . . . . 3
. . . . . F . . F .
. . . . . . . . . F
. . . . . . . . . .
2 . . . . . . . . .
. . . . . F F . . .
. F . . . F . . . .
. . . . . . . . . .

--- Step 1 ---
. . . F 1 . . . . .
. . . . . . . . . .
. . . . . . . . . .
. . . . . F . . F 3
. . . . . . . . . F
. . . . . . . . . .
. . . . . . . . . .
2 . . . . F F . . .
. F . . . F . . . .
. . . . . . . . . .

--- Step 2 ---
. . . 1 . . . . . .
. . . . . . . . . .
. . . . . . . . . .
. . . . . F . . F .
. . . . . . . . . 3
. . . . . . . . . .
. . . . . . . . . .
. . . . . F F . . .
2 F . . . F . . . .
. . . . . . . . . .

--- Step 3 ---
. . . . . . . . . .
. . . 1 . . . . . .
. . . . . . . . . .
. . . . . F . . F 3
. . . . . . . . . .
. . . . . . . . . .
. . . . . . . . . .
. . . . . F F . . .
. 2 . . . F . . . .
. . . . . . . . . .

--- Step 4 ---
. . . . . . . . . .
. . . . . . . . . .
. . . 1 . . . . . .
. . . . . F . . 3 .
. . . 